
# Parkinson’s Disease — UPDRS Regression & Classification (Robust Template)

This notebook is **flexible**: it will automatically adapt to the dataset you place in `data/`.

It supports two common datasets:

1. **Telemonitoring** (contains `total_UPDRS` and/or `motor_UPDRS`) → **Regression**
2. **UCI Parkinsons** (contains `status` 0/1) → **Classification**

The notebook will:
- Load whichever CSV is in `data/` (you can have multiple; it will ask you to choose if more than one is found).
- Clean & preprocess features.
- Run **regression** if it finds `total_UPDRS` or `motor_UPDRS`.
- Run **classification** if it finds `status` (or it will auto-create `high_updrs` from `total_UPDRS` if you want a classification target).
- Save metrics under `results/`.


In [ ]:

import os
import json
import glob
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# For plotting
import matplotlib.pyplot as plt

# Ensure folders exist
Path('results').mkdir(exist_ok=True, parents=True)
Path('data').mkdir(exist_ok=True, parents=True)

print('✅ Folders ready: data/, results/')


In [ ]:

# === Discover CSV files in data/ ===
csvs = sorted(glob.glob('data/*.csv'))
if not csvs:
    raise FileNotFoundError('No CSV found in data/. Please add your dataset, e.g., data/parkinsons.csv')

if len(csvs) == 1:
    csv_path = csvs[0]
    print(f'Found dataset: {csv_path}')
else:
    # If multiple, pick the first but print options
    print('Multiple CSV files found:')
    for i, p in enumerate(csvs, 1):
        print(f'{i}. {p}')
    csv_path = csvs[0]
    print(f'Using the first by default: {csv_path}')
    
df = pd.read_csv(csv_path)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()


In [ ]:

# === Determine available targets ===
targets = {
    'total_UPDRS': 'regression',
    'motor_UPDRS': 'regression',
    'status': 'classification'
}

available = [t for t in targets if t in df.columns]
print('Detected potential targets:', available)

task = None
REG_TARGET = None
CLF_TARGET = None

# Priority: Try regression on total_UPDRS first, else motor_UPDRS.
if 'total_UPDRS' in available:
    task = 'regression'
    REG_TARGET = 'total_UPDRS'
elif 'motor_UPDRS' in available:
    task = 'regression'
    REG_TARGET = 'motor_UPDRS'
elif 'status' in available:
    task = 'classification'
    CLF_TARGET = 'status'
else:
    # If no targets found, but total_UPDRS exists under a different case or spacing, try to infer
    # Also as a fallback: if there is a numeric column that looks like UPDRS, try to suggest
    # Otherwise, create a classification label from a numeric column if the user wants
    print('No standard target columns found.')
    # Try case-insensitive search
    lower_map = {c.lower(): c for c in df.columns}
    if 'total_updrs' in lower_map:
        REG_TARGET = lower_map['total_updrs']
        task = 'regression'
        print(f"Inferred target column (case-insensitive): {REG_TARGET}")
    elif 'motor_updrs' in lower_map:
        REG_TARGET = lower_map['motor_updrs']
        task = 'regression'
        print(f"Inferred target column (case-insensitive): {REG_TARGET}")
    else:
        # As a last resort: create a binary target from a numeric column if it exists
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_cols:
            base = numeric_cols[0]
            median_val = df[base].median()
            df['auto_label'] = (df[base] > median_val).astype(int)
            task = 'classification'
            CLF_TARGET = 'auto_label'
            print(f'Created fallback classification label auto_label from numeric column: {base} (threshold=median={median_val:.3f})')
        else:
            raise ValueError('Could not determine task because no valid targets or numeric columns were found. Please ensure columns include total_UPDRS, motor_UPDRS, or status.')

print('Task:', task)
print('REG_TARGET:', REG_TARGET)
print('CLF_TARGET:', CLF_TARGET)


In [ ]:

# === Build feature matrix X and target y ===
def build_xy(df, target):
    # Drop obvious ID/name-like columns if present
    id_like = [c for c in df.columns if c.lower() in {'name','id','subject','patient','record'}]
    drop_cols = list(set(id_like + [target]))
    X = df.drop(columns=drop_cols, errors='ignore')
    y = df[target].values
    return X, y

if task == 'regression':
    X, y = build_xy(df.copy(), REG_TARGET)
elif task == 'classification':
    X, y = build_xy(df.copy(), CLF_TARGET)

print('X shape:', X.shape)
print('y shape:', y.shape)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:

# === Preprocess numeric / categorical ===
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
    # (Optional) OneHotEncoder can be added if categorical columns exist
])

pre = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ],
    remainder='drop'
)

if task == 'regression':
    model = RandomForestRegressor(random_state=42, n_estimators=300)
else:
    model = RandomForestClassifier(random_state=42, n_estimators=300)

pipe = Pipeline(steps=[('preprocess', pre),
                      ('model', model)])

pipe.fit(X_train, y_train)

print('Model trained.')


In [ ]:

metrics = {}
if task == 'regression':
    preds = pipe.predict(X_test)
    metrics['MAE'] = float(mean_absolute_error(y_test, preds))
    metrics['RMSE'] = float(mean_squared_error(y_test, preds, squared=False))
    metrics['R2'] = float(r2_score(y_test, preds))
    print('Regression metrics:', metrics)
else:
    preds = pipe.predict(X_test)
    metrics['Accuracy'] = float(accuracy_score(y_test, preds))
    metrics['Precision'] = float(precision_score(y_test, preds, zero_division=0))
    metrics['Recall'] = float(recall_score(y_test, preds, zero_division=0))
    metrics['F1'] = float(f1_score(y_test, preds, zero_division=0))
    print('Classification metrics:', metrics)

# Save metrics
result_file = Path('results') / f'{task}_metrics.json'
with open(result_file, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print(f'Saved metrics to {result_file}')


In [ ]:

# === Feature importance plot (if supported) ===
try:
    model_step = pipe.named_steps['model']
    if hasattr(model_step, 'feature_importances_'):
        # Get transformed feature names (approximate: numeric + categorical)
        feat_names = num_cols + cat_cols
        importances = model_step.feature_importances_
        # Match lengths defensively
        k = min(len(importances), len(feat_names))
        imp = pd.Series(importances[:k], index=feat_names[:k]).sort_values(ascending=False)[:20]
        
        plt.figure(figsize=(8, 6))
        imp[::-1].plot(kind='barh')
        plt.title('Top Feature Importances')
        plt.tight_layout()
        plt.savefig('results/feature_importance.png', dpi=150)
        plt.show()
        print('Saved plot to results/feature_importance.png')
    else:
        print('Model has no feature_importances_. Skipping plot.')
except Exception as e:
    print('Feature importance plotting skipped:', e)


In [ ]:

import joblib
model_path = Path('results') / f'{task}_pipeline.joblib'
joblib.dump(pipe, model_path)
print(f'Saved trained pipeline to {model_path}')
